In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.datasets import load_gbsg2
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                            ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
import statsmodels.api as sm
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
 
from sklearn.preprocessing import PowerTransformer

In [2]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [3]:
# Need to choose patient_id from OUS_D2 in response_OUS
data = list(OUS_D2['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D2 with response_OUS
clinical_train = pd.merge(OUS_D2, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS', 'LRC', 'event_LRC'])]

In [4]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

In [5]:
# Check null values in D2 
clinical_train.isnull().sum().sum()

0

In [6]:
""" Takes too long time 
# Check collinearity for OUS data 
df = clinical_train

# Create a correlation matrix
correlation_matrix = df.corr()

plt.figure(figsize=(15, 12))

# Plot a heatmap
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)
plt.show()
""" 

" Takes too long time \n# Check collinearity for OUS data \ndf = clinical_train\n\n# Create a correlation matrix\ncorrelation_matrix = df.corr()\n\nplt.figure(figsize=(15, 12))\n\n# Plot a heatmap\nsns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)\nplt.show()\n"

### COX assumption in Train data

In [7]:
df = pd.DataFrame(clinical_train)

# Fit Cox proportional hazards model
cph = CoxPHFitter(penalizer=0.00001)
cph.fit(df, duration_col='DFS', event_col='event_DFS')

# Perform the proportional hazards assumption test
results = proportional_hazard_test(cph, df)
print(results)

<lifelines.CoxPHFitter: fitted with 139 total observations, 71 right-censored observations>

<lifelines.StatisticalResult: proportional_hazard_test>
    time_transform = rank
 null_distribution = chi squared
degrees_of_freedom = 1
             model = <lifelines.CoxPHFitter: fitted with 139 total observations, 71 right-censored observations>
         test_name = proportional_hazard_test

---
                                                       test_statistic    p  -log2(p)
LBP_003_CT                                                       0.00 0.95      0.08
LBP_003_PET                                                      0.00 0.99      0.01
LBP_012_CT                                                       0.01 0.91      0.13
LBP_012_PET                                                      0.00 0.98      0.03
LBP_021_CT                                                       0.01 0.94      0.09
LBP_021_PET                                                      0.02 0.90      0.16
LBP_030_CT                                                       0.00 0.97      0.04
LBP_030_PET       

In [8]:
# Access the summary table
summary_table = results.summary

# Filter columns based on p-values lower than 0.05
significant_columns = summary_table[summary_table['p'] < 0.05].index

# Display significant columns
print("Columns with p-values < 0.05:")
print(significant_columns)

Columns with p-values < 0.05:
Index([], dtype='object')


In [9]:
df = pd.DataFrame(clinical_train)

# Fit Cox proportional hazards model
cph = CoxPHFitter(penalizer=0.1)
cph.fit(df, duration_col='DFS', event_col='event_DFS')

# Perform the proportional hazards assumption test
results = proportional_hazard_test(cph, df)
print(results)

<lifelines.CoxPHFitter: fitted with 139 total observations, 71 right-censored observations>

<lifelines.StatisticalResult: proportional_hazard_test>
    time_transform = rank
 null_distribution = chi squared
degrees_of_freedom = 1
             model = <lifelines.CoxPHFitter: fitted with 139 total observations, 71 right-censored observations>
         test_name = proportional_hazard_test

---
                                                       test_statistic      p  -log2(p)
LBP_003_CT                                                       0.20   0.65      0.62
LBP_003_PET                                                      0.43   0.51      0.96
LBP_012_CT                                                       0.19   0.66      0.59
LBP_012_PET                                                      0.18   0.67      0.57
LBP_021_CT                                                       0.94   0.33      1.59
LBP_021_PET                                                      0.01   0.93      0.10
LBP_030_CT                                                       0.00   0.94      0.08
LB

In [10]:
# Access the summary table
summary_table = results.summary

# Filter columns based on p-values lower than 0.05
significant_columns = summary_table[summary_table['p'] < 0.05].index

# Display significant columns
print("Columns with p-values < 0.05:")
print(significant_columns)

Columns with p-values < 0.05:
Index(['glszm_GrayLevelNonUniformity_PET_c04',
       'glszm_LargeAreaEmphasis_CT_c16',
       'glszm_LargeAreaLowGrayLevelEmphasis_CT_c16',
       'glszm_ZoneVariance_CT_c16', 'ngtdm_Busyness_d_1_PET_b2'],
      dtype='object')


###### penalizer values essentially result in same result 

## Test dataset: MAASTRO 

In [11]:
(MAASTRO_D2['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [12]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [13]:
# need to choose patient_id from MAASTRO_D2 in response_MAASTRO
data = list(MAASTRO_D2['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 
response_MAASTRO

,patient_id,OS,OS_event,LRC,LRC_event,DFS,DFS_event
0,1,62.43,0.0,62.43,0.0,62.43,0.0
1,2,60.00,0.0,60.00,0.0,60.00,0.0
2,3,44.43,1.0,8.83,1.0,8.83,1.0
3,4,37.20,1.0,19.37,0.0,19.73,1.0
5,6,59.23,0.0,59.23,0.0,59.23,0.0
...,...,...,...,...,...,...,...
109,110,19.00,1.0,13.27,1.0,13.27,1.0
110,111,85.87,1.0,82.83,0.0,85.87,1.0
111,112,42.87,0.0,42.87,0.0,42.87,0.0
112,113,58.93,0.0,58.93,0.0,58.93,0.0


In [14]:
# Merge MAASTRO_D2 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D2, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'OS_event', 'LRC', 'LRC_event'])]
clinical_test

,patient_id,shape_Elongation,shape_Flatness,shape_LeastAxisLength,shape_MajorAxisLength,shape_Maximum2DDiameterColumn,shape_Maximum2DDiameterRow,shape_Maximum2DDiameterSlice,shape_Maximum3DDiameter,shape_MeshVolume,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,DFS,DFS_event
0,1,0.765178,0.610062,30.504395,50.002093,52.886671,58.000000,47.759816,58.864251,38067.208333,...,0.028808,0.837387,0.000026,0.001705,0.122209,0.000026,0.008291,0.000341,62.43,0.0
1,2,0.776540,0.504616,21.069386,41.753334,44.922155,44.294469,44.147480,48.723711,17870.791667,...,0.049615,0.806842,0.000167,0.002958,0.128976,0.000167,0.008148,0.000335,60.00,0.0
2,3,0.697164,0.478604,21.238275,44.375483,48.466483,45.891176,37.656341,48.969378,17426.500000,...,0.019514,0.830272,0.000057,0.001831,0.137282,0.000000,0.008641,0.000229,8.83,1.0
3,4,0.574636,0.446059,20.570459,46.115989,42.544095,47.507894,33.837849,55.226805,12463.791667,...,0.047155,0.760949,0.000000,0.003597,0.171595,0.000080,0.013187,0.000240,19.73,1.0
4,6,0.633419,0.480378,26.130155,54.394967,66.030296,56.320511,45.398238,67.089492,28300.125000,...,0.033990,0.824865,0.000000,0.002116,0.128134,0.000035,0.009379,0.000529,59.23,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,110,0.882411,0.577884,19.774388,34.218615,41.785165,40.261644,35.805028,43.520110,12822.458333,...,0.017489,0.834590,0.000000,0.001321,0.137194,0.000078,0.008706,0.000078,13.27,1.0
95,111,0.535802,0.455642,23.259099,51.046869,51.264022,51.264022,33.600595,52.440442,18368.291667,...,0.029979,0.821431,0.000000,0.000543,0.137620,0.000054,0.008907,0.000489,85.87,1.0
96,112,0.716610,0.631485,31.838169,50.417953,46.324939,56.035703,57.070132,57.671483,35384.541667,...,0.017354,0.859957,0.000000,0.000988,0.115099,0.000028,0.005700,0.000141,42.87,0.0
97,113,0.665145,0.628338,28.213278,44.901412,50.289164,50.596443,35.777088,51.478151,26180.208333,...,0.025399,0.847908,0.000000,0.001487,0.117654,0.000000,0.005759,0.000038,58.93,0.0


In [15]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [16]:
# Some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,shape_Elongation,shape_Flatness,shape_LeastAxisLength,shape_MajorAxisLength,shape_Maximum2DDiameterColumn,shape_Maximum2DDiameterRow,shape_Maximum2DDiameterSlice,shape_Maximum3DDiameter,shape_MeshVolume,shape_MinorAxisLength,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,DFS,DFS_event


In [17]:
# X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS'])]

# y 
y = clinical_train.loc[:, ['DFS', 'event_DFS']]

In [18]:
# Set lower, upper time point and times for IBS calculation later 
lower, upper = np.percentile(y['DFS'], [10, 90])
times = np.arange(lower, upper)

# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [19]:
# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

X_train:  (139, 374)
y_train:  (139,)


In [20]:
clinical_test.rename(columns = {'DFS_event' : 'event_DFS'}, inplace = True)

In [21]:
# Set X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'event_DFS'])]

# Set y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['DFS', 'event_DFS']]

# Change y_MAASTRO into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_DFS'], y_MAASTRO['DFS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

(99, 376)

In [22]:
# VIF dataframe 
vif_data = pd.DataFrame() 
vif_data["feature"] = X.columns 
  
# calculating VIF for each feature 
vif_data["VIF"] = [variance_inflation_factor(X.values, i) 
                          for i in range(len(X.columns))] 

vif_data[vif_data['VIF'] > 100000]

,feature,VIF
0,shape_Elongation,2.096355e+11
1,shape_Flatness,3.609232e+11
2,shape_LeastAxisLength,2.691126e+12
3,shape_MajorAxisLength,7.524811e+12
4,shape_Maximum2DDiameterColumn,2.124339e+13
...,...,...
369,LBP_111_PET,5.922282e+11
370,LBP_120_PET,3.305758e+11
371,LBP_201_PET,1.508996e+12
372,LBP_210_PET,1.369708e+12


# Yeo-Johnson Transformation

In [23]:
original_X = X.copy()

In [24]:
# Standardize the data 
## Save the column and index 
X_columns = X.columns 
X_index = X.index

# Standardize non-categorical and then concat with the categorical
pt = PowerTransformer(method='yeo-johnson')
X_std = pt.fit_transform(X)
X_std = pd.DataFrame(X_std, columns=X_columns, index=X_index)

In [25]:
# Standardize the numeric part 
X_MAASTRO_column = X_MAASTRO.columns
X_MAASTRO_index = X_MAASTRO.index

X_MAASTRO_std = pt.transform(X_MAASTRO)

# Change the standardized part into a dataframe 
X_MAASTRO_std = pd.DataFrame(X_MAASTRO_std, columns=X_MAASTRO_column, index=X_MAASTRO_index)

In [26]:
# Saving the data 
X_new = X 
X_new_std = X_std 
MAASTRO_new = X_MAASTRO 
MAASTRO_new_std = X_MAASTRO_std 

# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [27]:
# Setting the y format for skf below  
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-15 13:59:58,280] A new study created in memory with name: no-name-9356ed6b-3a51-4dc3-b783-0b2438c9cd00


  0%|          | 0/1 [00:00<?, ?it/s]

[W 2024-04-15 14:00:00,321] Trial 0 failed with parameters: {} because of the following error: LinAlgError('Matrix is singular.').
Traceback (most recent call last):
  File "/Users/minjeongcheon/opt/anaconda3/lib/python3.9/site-packages/optuna/study/_optimize.py", line 200, in _run_trial
    value_or_values = func(trial)
  File "/var/folders/64/jkqp6xyx2hj50dmd2pqfm3780000gn/T/ipykernel_68893/3148227347.py", line 62, in objective
    model.fit(X_train_std, y_train)
  File "/Users/minjeongcheon/opt/anaconda3/lib/python3.9/site-packages/sksurv/linear_model/coxph.py", line 449, in fit
    delta = solve(
  File "/Users/minjeongcheon/opt/anaconda3/lib/python3.9/site-packages/scipy/linalg/_basic.py", line 220, in solve
    _solve_check(n, info)
  File "/Users/minjeongcheon/opt/anaconda3/lib/python3.9/site-packages/scipy/linalg/_basic.py", line 29, in _solve_check
    raise LinAlgError('Matrix is singular.')
numpy.linalg.LinAlgError: Matrix is singular.
[W 2024-04-15 14:00:00,328] Trial 0 fai

LinAlgError: Matrix is singular.

In [28]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

ValueError: No trials are completed yet.

In [29]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

ValueError: No trials are completed yet.

#### Test

In [30]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [31]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

LinAlgError: Matrix is singular.

In [32]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [33]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

NameError: name 'c_index' is not defined

### 2. CoxnetSurvivalAnalysis - Ridge

#### Train

In [34]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-15 14:02:31,947] A new study created in memory with name: no-name-327504a9-fba3-466b-acc5-3cc6e5183d04


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.5756972111553785
Fold 2 C-index: 0.7073643410852714
Fold 3 C-index: 0.5702127659574469
Fold 4 C-index: 0.6653992395437263


[I 2024-04-15 14:02:39,309] A new study created in memory with name: no-name-f7a3d3a2-0edb-4cc0-aa25-1c86b396262c


Fold 5 C-index: 0.6566523605150214
[I 2024-04-15 14:02:39,298] Trial 0 finished with value: 0.635065183651369 and parameters: {}. Best is trial 0 with value: 0.635065183651369.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.635065183651369], datetime_start=datetime.datetime(2024, 4, 15, 14, 2, 31, 983955), datetime_complete=datetime.datetime(2024, 4, 15, 14, 2, 39, 297405), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.635065183651369


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.24724709663544278
Fold 2 IBS: 0.23203988184644345
Fold 3 IBS: 0.22898186778522736
Fold 4 IBS: 0.24197476559442144
Fold 5 IBS: 0.22939558964342896
[I 2024-04-15 14:02:47,032] Trial 0 finished with value: 0.23592784030099279 and parameters: {}. Best is trial 0 with value: 0.23592784030099279.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.23592784030099279], datetime_start=datetime.datetime(2024, 4, 15, 14, 2, 39, 348088), datetime_complete=datetime.datetime(2024, 4, 15, 14, 2, 47, 31931), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.23592784030099279


In [35]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [36]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.635
train_ibs:  0.236


#### Test

In [37]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [38]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.53


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.229


In [39]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [40]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-15 14:02:47,272] A new study created in memory with name: no-name-b704c195-c903-4a42-a43c-03fc841c099c


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.5737051792828686
Fold 2 C-index: 0.6201550387596899
Fold 3 C-index: 0.37446808510638296
Fold 4 C-index: 0.6387832699619772


[I 2024-04-15 14:02:57,922] A new study created in memory with name: no-name-6a4700b5-4577-4498-bae0-debcf5571b72


Fold 5 C-index: 0.6051502145922747
[I 2024-04-15 14:02:57,909] Trial 0 finished with value: 0.5624523575406387 and parameters: {}. Best is trial 0 with value: 0.5624523575406387.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.5624523575406387], datetime_start=datetime.datetime(2024, 4, 15, 14, 2, 47, 307345), datetime_complete=datetime.datetime(2024, 4, 15, 14, 2, 57, 909033), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.5624523575406387


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.38515817356172605
Fold 2 IBS: 0.31542055044032496
Fold 3 IBS: 0.46463727872638066
Fold 4 IBS: 0.27876072700041465
Fold 5 IBS: 0.2803374283965891
[I 2024-04-15 14:03:09,129] Trial 0 finished with value: 0.34486283162508713 and parameters: {}. Best is trial 0 with value: 0.34486283162508713.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.34486283162508713], datetime_start=datetime.datetime(2024, 4, 15, 14, 2, 57, 954561), datetime_complete=datetime.datetime(2024, 4, 15, 14, 3, 9, 128136), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.34486283162508713


In [41]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [42]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.562
train_ibs:  0.345


#### Test

In [43]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [44]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.477


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.464


In [45]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [46]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-15 14:03:11,098] A new study created in memory with name: no-name-882406fa-a761-41fe-950f-1a87d291847d


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5617529880478087
Fold 2 C-index: 0.6162790697674418
Fold 3 C-index: 0.39148936170212767
Fold 4 C-index: 0.6083650190114068
Fold 5 C-index: 0.6051502145922747
[I 2024-04-15 14:03:20,751] Trial 0 finished with value: 0.5566073306242119 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.5566073306242119.
Fold 1 C-index: 0.5258964143426295
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.46382978723404256
Fold 4 C-index: 0.6007604562737643
Fold 5 C-index: 0.5879828326180258
[I 2024-04-15 14:03:30,879] Trial 1 finished with value: 0.5752287818146227 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.5752287818146227.
Fold 1 C-index: 0.5258964143426295
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.46808510638297873
Fold 4 C-index: 0.596958174904943
Fold 5 C-index: 0.5879828326180258
[I 2024-04-15 14:03:38,799] Trial 2 finished with value: 0.5753193893706456 and parameters: {'l1_ratio': 0.22692876841884668

Fold 5 C-index: 0.5879828326180258
[I 2024-04-15 14:06:39,042] Trial 23 finished with value: 0.5870532612611276 and parameters: {'l1_ratio': 0.17128591911951985}. Best is trial 12 with value: 0.6412790481054278.
Fold 1 C-index: 0.5258964143426295
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.4553191489361702
Fold 4 C-index: 0.6007604562737643
Fold 5 C-index: 0.5965665236051502
[I 2024-04-15 14:06:48,302] Trial 24 finished with value: 0.575243392352473 and parameters: {'l1_ratio': 0.31612098478358897}. Best is trial 12 with value: 0.6412790481054278.
Fold 1 C-index: 0.4860557768924303
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6920152091254753
Fold 5 C-index: 0.6695278969957081
[I 2024-04-15 14:06:58,854] Trial 25 finished with value: 0.6213950858555679 and parameters: {'l1_ratio': 0.08890608406899088}. Best is trial 12 with value: 0.6412790481054278.
Fold 1 C-index: 0.5258964143426295
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index:

Fold 1 C-index: 0.5776892430278885
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6958174904942965
Fold 5 C-index: 0.6609442060085837
[I 2024-04-15 14:10:04,022] Trial 47 finished with value: 0.6387654971589989 and parameters: {'l1_ratio': 0.04717599012383908}. Best is trial 12 with value: 0.6412790481054278.
Fold 1 C-index: 0.5139442231075697
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.5817490494296578
Fold 5 C-index: 0.5879828326180258
[I 2024-04-15 14:10:20,680] Trial 48 finished with value: 0.5886105302838958 and parameters: {'l1_ratio': 0.18099861440113463}. Best is trial 12 with value: 0.6412790481054278.
Fold 1 C-index: 0.49800796812749004
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.688212927756654
Fold 5 C-index: 0.6695278969957081
[I 2024-04-15 14:10:28,888] Trial 49 finished with value: 0.6230250678288156 and parameters: {'l1_ratio': 0.1175964763683

...

  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 IBS: 0.3918623450099054
Fold 2 IBS: 0.3054662030003641
Fold 3 IBS: 0.40752625572323914
Fold 4 IBS: 0.2666150439066279
Fold 5 IBS: 0.28293007057112657
[I 2024-04-15 14:21:58,198] Trial 0 finished with value: 0.33087998364225263 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.33087998364225263.
Fold 1 IBS: 0.3998476080259402
Fold 2 IBS: 0.2265618087310044
Fold 3 IBS: 0.3475303299925298
Fold 4 IBS: 0.29710817476625395
Fold 5 IBS: 0.27023143183001885
[I 2024-04-15 14:22:12,053] Trial 1 finished with value: 0.30825587066914945 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.30825587066914945.
Fold 1 IBS: 0.4005990952555494
Fold 2 IBS: 0.22741753924120442
Fold 3 IBS: 0.33864421758995367
Fold 4 IBS: 0.2993388587877366
Fold 5 IBS: 0.2634908243326706
[I 2024-04-15 14:22:24,959] Trial 2 finished with value: 0.30589810704142295 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 2 with value: 0.30589810704142295.

Fold 1 IBS: 0.3913430452965101
Fold 2 IBS: 0.22991305573774998
Fold 3 IBS: 0.22878890533246202
Fold 4 IBS: 0.23805110419288075
Fold 5 IBS: 0.22694636185590233
[I 2024-04-15 14:30:41,877] Trial 25 finished with value: 0.263008494483101 and parameters: {'l1_ratio': 0.08890608406899088}. Best is trial 12 with value: 0.23515110302320735.
Fold 1 IBS: 0.4008483195893078
Fold 2 IBS: 0.22729127829046636
Fold 3 IBS: 0.3396431684651452
Fold 4 IBS: 0.2992938684480467
Fold 5 IBS: 0.26441776623072394
[I 2024-04-15 14:31:08,281] Trial 26 finished with value: 0.306298880204738 and parameters: {'l1_ratio': 0.23521758130887002}. Best is trial 12 with value: 0.23515110302320735.
Fold 1 IBS: 0.39309430990207567
Fold 2 IBS: 0.3100160996593142
Fold 3 IBS: 0.4236545872482596
Fold 4 IBS: 0.2695880122170518
Fold 5 IBS: 0.2826521039656795
[I 2024-04-15 14:31:36,996] Trial 27 finished with value: 0.3358010225984761 and parameters: {'l1_ratio': 0.8345729974660353}. Best is trial 12 with value: 0.2351511030232073

Fold 1 IBS: 0.39322003858934845
Fold 2 IBS: 0.2987344065368188
Fold 3 IBS: 0.39044818172621804
Fold 4 IBS: 0.27570374849433044
Fold 5 IBS: 0.28089377377571967
[I 2024-04-15 14:37:58,697] Trial 50 finished with value: 0.3278000298244871 and parameters: {'l1_ratio': 0.556147470734502}. Best is trial 30 with value: 0.2338861317669947.
Fold 1 IBS: 0.24527167584214757
Fold 2 IBS: 0.23043201333842706
Fold 3 IBS: 0.22882959418766094
Fold 4 IBS: 0.23886096933818587
Fold 5 IBS: 0.2275108188111229
[I 2024-04-15 14:38:11,697] Trial 51 finished with value: 0.23418101430350885 and parameters: {'l1_ratio': 0.06521384802113693}. Best is trial 30 with value: 0.2338861317669947.
Fold 1 IBS: 0.24563902210807548
Fold 2 IBS: 0.23076318528471024
Fold 3 IBS: 0.228857861655577
Fold 4 IBS: 0.23942629816028696
Fold 5 IBS: 0.22788081987002892
[I 2024-04-15 14:38:24,597] Trial 52 finished with value: 0.2345134374157357 and parameters: {'l1_ratio': 0.05081950410384198}. Best is trial 30 with value: 0.233886131766

Fold 1 IBS: 0.24502298791627394
Fold 2 IBS: 0.23019984637955831
Fold 3 IBS: 0.22881083058401447
Fold 4 IBS: 0.23848030996038092
Fold 5 IBS: 0.22725595943412547
[I 2024-04-15 14:43:53,869] Trial 75 finished with value: 0.23395398685487062 and parameters: {'l1_ratio': 0.07563581400941731}. Best is trial 65 with value: 0.2338114901918404.
Fold 1 IBS: 0.39566788032604905
Fold 2 IBS: 0.22905991597084543
Fold 3 IBS: 0.22873248532776408
Fold 4 IBS: 0.2369097692969695
Fold 5 IBS: 0.2260452207998657
[I 2024-04-15 14:44:05,861] Trial 76 finished with value: 0.2632830543442988 and parameters: {'l1_ratio': 0.13120285166575224}. Best is trial 65 with value: 0.2338114901918404.
Fold 1 IBS: 0.399514089081406
Fold 2 IBS: 0.2263144117931058
Fold 3 IBS: 0.3514093726185773
Fold 4 IBS: 0.2973571561270843
Fold 5 IBS: 0.27140764137209317
[I 2024-04-15 14:44:19,159] Trial 77 finished with value: 0.3092005341984533 and parameters: {'l1_ratio': 0.3048122805041498}. Best is trial 65 with value: 0.23381149019184

In [47]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [48]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.641
train_ibs:  0.234


#### Test

In [49]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [50]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.025939717088620355)

test_cindex : 0.527


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.08286735522305856)

test_ibs:  0.229


In [51]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [52]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-15 14:48:55,041] A new study created in memory with name: no-name-b2f6658c-f680-4909-8747-2cebd9dd5d81


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6135458167330677
Fold 2 C-index: 0.7558139534883721
Fold 3 C-index: 0.6425531914893617
Fold 4 C-index: 0.7186311787072244
Fold 5 C-index: 0.6523605150214592
[I 2024-04-15 14:49:33,673] Trial 0 finished with value: 0.676580931087897 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.676580931087897.
Fold 1 C-index: 0.5537848605577689
Fold 2 C-index: 0.6744186046511628
Fold 3 C-index: 0.6893617021276596
Fold 4 C-index: 0.6273764258555133
Fold 5 C-index: 0.5278969957081545
[I 2024-04-15 14:49:40,582] Trial 1 finished with value: 0.6145677177800518 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, 'm

Fold 5 C-index: 0.6824034334763949
[I 2024-04-15 14:51:56,218] Trial 15 finished with value: 0.7109217554965979 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 7, 'min_samples_leaf': 16, 'max_depth': 6, 'n_estimators': 96, 'oob_score': True, 'max_samples': 0.8147833133306369, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.2047601680710191, 'warm_start': True}. Best is trial 15 with value: 0.7109217554965979.
Fold 1 C-index: 0.3804780876494024
Fold 2 C-index: 0.810077519379845
Fold 3 C-index: 0.7936170212765957
Fold 4 C-index: 0.5836501901140685
Fold 5 C-index: 0.6351931330472103
[I 2024-04-15 14:51:56,561] Trial 16 finished with value: 0.6406031902934244 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 2, 'min_samples_leaf': 18, 'max_depth': 6, 'n_estimators': 5, 'oob_score': True, 'max_samples': 0.8484814588885312, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.3777228897992014, 'warm_start': True}. Best is trial 15 with value: 0.7109217554965979.


Fold 1 C-index: 0.549800796812749
Fold 2 C-index: 0.8255813953488372
Fold 3 C-index: 0.8680851063829788
Fold 4 C-index: 0.8288973384030418
Fold 5 C-index: 0.8111587982832618
[I 2024-04-15 14:52:17,457] Trial 30 finished with value: 0.7767046870461737 and parameters: {'min_samples_split': 11, 'max_leaf_nodes': 9, 'min_samples_leaf': 10, 'max_depth': 12, 'n_estimators': 122, 'oob_score': True, 'max_samples': 0.6865809791097957, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.03407840564085754, 'warm_start': True}. Best is trial 30 with value: 0.7767046870461737.
Fold 1 C-index: 0.549800796812749
Fold 2 C-index: 0.8294573643410853
Fold 3 C-index: 0.8723404255319149
Fold 4 C-index: 0.8288973384030418
Fold 5 C-index: 0.8111587982832618
[I 2024-04-15 14:52:19,087] Trial 31 finished with value: 0.7783309446744106 and parameters: {'min_samples_split': 11, 'max_leaf_nodes': 9, 'min_samples_leaf': 10, 'max_depth': 12, 'n_estimators': 128, 'oob_score': True, 'max_samples': 0.686417287257043

...

Fold 1 IBS: 0.223671117899065
Fold 2 IBS: 0.18242513973526725
Fold 3 IBS: 0.2389733570256732
Fold 4 IBS: 0.21132681423071092
Fold 5 IBS: 0.2129200403370242
[I 2024-04-15 15:29:42,584] Trial 32 finished with value: 0.2138632938455481 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 11, 'min_samples_leaf': 8, 'max_depth': 14, 'n_estimators': 294, 'oob_score': True, 'max_samples': 0.9673759674409311, 'max_features': None, 'min_weight_fraction_leaf': 0.27417727195073677}. Best is trial 32 with value: 0.2138632938455481.
Fold 1 IBS: 0.2573672754798652
Fold 2 IBS: 0.1902152831375234
Fold 3 IBS: 0.2272585883751577
Fold 4 IBS: 0.22496428138050273
Fold 5 IBS: 0.2133258411787376
[I 2024-04-15 15:30:53,755] Trial 33 finished with value: 0.2226262539103573 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 13, 'min_samples_leaf': 13, 'max_depth': 13, 'n_estimators': 304, 'oob_score': True, 'max_samples': 0.9964442664551478, 'max_features': None, 'min_weight_fraction_leaf': 0.185

Fold 1 IBS: 0.23136140224576346
Fold 2 IBS: 0.1874927380948624
Fold 3 IBS: 0.2417750554443432
Fold 4 IBS: 0.2265069844402473
Fold 5 IBS: 0.2218959262967509
[I 2024-04-15 15:37:54,995] Trial 48 finished with value: 0.22180642130439346 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 18, 'min_samples_leaf': 5, 'max_depth': 15, 'n_estimators': 354, 'oob_score': True, 'max_samples': 0.9047670799898727, 'max_features': None, 'min_weight_fraction_leaf': 0.31310643491678164}. Best is trial 32 with value: 0.2138632938455481.
Fold 1 IBS: 0.23542420632037062
Fold 2 IBS: 0.21022618815946087
Fold 3 IBS: 0.24786241576097612
Fold 4 IBS: 0.24693634633915879
Fold 5 IBS: 0.2275366959349153
[I 2024-04-15 15:38:03,793] Trial 49 finished with value: 0.23359717050297632 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 16, 'min_samples_leaf': 20, 'max_depth': 13, 'n_estimators': 126, 'oob_score': True, 'max_samples': 0.7434493023402162, 'max_features': None, 'min_weight_fraction_leaf': 0

Fold 1 IBS: 0.23011155909129694
Fold 2 IBS: 0.18057364325153813
Fold 3 IBS: 0.23750151803668482
Fold 4 IBS: 0.21433948253554103
Fold 5 IBS: 0.21274097244510462
[I 2024-04-15 15:48:59,831] Trial 64 finished with value: 0.2150534350720331 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 8, 'min_samples_leaf': 4, 'max_depth': 17, 'n_estimators': 249, 'oob_score': True, 'max_samples': 0.9368121127684691, 'max_features': None, 'min_weight_fraction_leaf': 0.24195064098385735}. Best is trial 32 with value: 0.2138632938455481.
Fold 1 IBS: 0.22994947133324334
Fold 2 IBS: 0.1928070595691341
Fold 3 IBS: 0.24608508712178995
Fold 4 IBS: 0.22580322695561086
Fold 5 IBS: 0.2205885428532394
[I 2024-04-15 15:49:32,387] Trial 65 finished with value: 0.2230466775666035 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 9, 'min_samples_leaf': 3, 'max_depth': 13, 'n_estimators': 318, 'oob_score': True, 'max_samples': 0.8670083844909762, 'max_features': None, 'min_weight_fraction_leaf': 0.

Fold 1 IBS: 0.23374541142463764
Fold 2 IBS: 0.19941476694876598
Fold 3 IBS: 0.24370640781635464
Fold 4 IBS: 0.23069544004770481
Fold 5 IBS: 0.2221664668271361
[I 2024-04-15 15:57:22,555] Trial 80 finished with value: 0.22594569861291985 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 13, 'min_samples_leaf': 4, 'max_depth': 17, 'n_estimators': 303, 'oob_score': True, 'max_samples': 0.6585925147157417, 'max_features': None, 'min_weight_fraction_leaf': 0.2649590538103387}. Best is trial 32 with value: 0.2138632938455481.
Fold 1 IBS: 0.22301744452662423
Fold 2 IBS: 0.18261690178298304
Fold 3 IBS: 0.24046916847175354
Fold 4 IBS: 0.21144194863813523
Fold 5 IBS: 0.21301702675170428
[I 2024-04-15 15:58:10,653] Trial 81 finished with value: 0.21411249803424007 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 10, 'min_samples_leaf': 2, 'max_depth': 16, 'n_estimators': 273, 'oob_score': True, 'max_samples': 0.9753152468491701, 'max_features': None, 'min_weight_fraction_leaf'

Fold 1 IBS: 0.23083721945133412
Fold 2 IBS: 0.18442690744496573
Fold 3 IBS: 0.2372912055911599
Fold 4 IBS: 0.22822622495739184
Fold 5 IBS: 0.22366315254862057
[I 2024-04-15 16:07:51,287] Trial 96 finished with value: 0.22088894199869444 and parameters: {'min_samples_split': 11, 'max_leaf_nodes': 10, 'min_samples_leaf': 3, 'max_depth': 14, 'n_estimators': 231, 'oob_score': True, 'max_samples': 0.9522685325324347, 'max_features': None, 'min_weight_fraction_leaf': 0.32080383117786954}. Best is trial 91 with value: 0.21308616320039436.
Fold 1 IBS: 0.2265678317463007
Fold 2 IBS: 0.18314595561796107
Fold 3 IBS: 0.2376073017283759
Fold 4 IBS: 0.21337826003109
Fold 5 IBS: 0.21787901230329237
[I 2024-04-15 16:08:15,497] Trial 97 finished with value: 0.21571567228540403 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 11, 'min_samples_leaf': 10, 'max_depth': 17, 'n_estimators': 174, 'oob_score': True, 'max_samples': 0.9189184529013809, 'max_features': None, 'min_weight_fraction_leaf': 

In [53]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [54]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.826
train_ibs:  0.213


#### Test

In [55]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

In [56]:
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])
 
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=13, max_features=None, max_leaf_nodes=10,
                     max_samples=0.750348183174299, min_samples_leaf=2,
                     min_samples_split=16,
                     min_weight_fraction_leaf=0.09298348876429541,
                     n_estimators=267, random_state=123, warm_start=True)

test_cindex:  0.516


RandomSurvivalForest(max_depth=5, max_features=None, max_leaf_nodes=13,
                     max_samples=0.9824332970054127, min_samples_split=10,
                     min_weight_fraction_leaf=0.296758543375246,
                     n_estimators=236, oob_score=True, random_state=123)

test_ibs:  0.258


In [57]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [58]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

In [59]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-15 16:09:26,885] A new study created in memory with name: no-name-82172fa8-bf44-48af-920d-28a3f3d9d084


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.4581673306772908
Fold 2 C-index: 0.7829457364341085
Fold 3 C-index: 0.8382978723404255
Fold 4 C-index: 0.688212927756654
Fold 5 C-index: 0.6909871244635193
[I 2024-04-15 16:09:28,593] Trial 0 finished with value: 0.6917221983343996 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.6917221983343996.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-15 16:09:33,600] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531}

Fold 1 C-index: 0.47808764940239046
Fold 2 C-index: 0.7596899224806202
Fold 3 C-index: 0.825531914893617
Fold 4 C-index: 0.6920152091254753
Fold 5 C-index: 0.6394849785407726
[I 2024-04-15 16:10:10,824] Trial 16 finished with value: 0.678961934888575 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 10, 'min_samples_leaf': 13, 'max_depth': 10, 'n_estimators': 95, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.884183184043622, 'min_weight_fraction_leaf': 0.10581506507448754}. Best is trial 8 with value: 0.7123582035090831.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-15 16:10:11,795] Trial 17 finished with value: 0.5 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 15, 'min_samples_leaf': 9, 'max_depth': 4, 'n_estimators': 162, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.37217752523657055, 'min_weight_fraction_leaf': 0.1994767874105

Fold 1 C-index: 0.47410358565737054
Fold 2 C-index: 0.7713178294573644
Fold 3 C-index: 0.8297872340425532
Fold 4 C-index: 0.6730038022813688
Fold 5 C-index: 0.6523605150214592
[I 2024-04-15 16:10:30,260] Trial 31 finished with value: 0.6801145932920232 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 4, 'min_samples_leaf': 8, 'max_depth': 13, 'n_estimators': 362, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.5225620293165905, 'min_weight_fraction_leaf': 0.0017285001202470562}. Best is trial 20 with value: 0.7286697682430041.
Fold 1 C-index: 0.49800796812749004
Fold 2 C-index: 0.7596899224806202
Fold 3 C-index: 0.825531914893617
Fold 4 C-index: 0.6387832699619772
Fold 5 C-index: 0.6437768240343348
[I 2024-04-15 16:10:32,469] Trial 32 finished with value: 0.6731579798996079 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 3, 'min_samples_leaf': 7, 'max_depth': 13, 'n_estimators': 372, 'oob_score': False, 'warm_start': True, 'max_feat

...

Fold 1 IBS: 0.23497354688702718
Fold 2 IBS: 0.1976009711176378
Fold 3 IBS: 0.20496554854896726
Fold 4 IBS: 0.22195679558988315
Fold 5 IBS: 0.22437156955219212
[I 2024-04-15 16:21:13,544] Trial 30 finished with value: 0.21677368633914149 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 15, 'min_samples_leaf': 13, 'max_depth': 12, 'n_estimators': 351, 'oob_score': False, 'warm_start': False, 'max_features': None, 'max_samples': 0.8683525444021144, 'min_weight_fraction_leaf': 0.10877158322577073}. Best is trial 30 with value: 0.21677368633914149.
Fold 1 IBS: 0.23741283004501632
Fold 2 IBS: 0.19660814788429976
Fold 3 IBS: 0.208320965032862
Fold 4 IBS: 0.22424875267364808
Fold 5 IBS: 0.22365262373167072
[I 2024-04-15 16:21:22,207] Trial 31 finished with value: 0.2180486638734994 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 15, 'min_samples_leaf': 13, 'max_depth': 12, 'n_estimators': 347, 'oob_score': False, 'warm_start': False, 'max_features': None, 'max_samples': 

Fold 1 IBS: 0.23322825812700776
Fold 2 IBS: 0.19980996308476648
Fold 3 IBS: 0.20366753393387363
Fold 4 IBS: 0.23327551134614005
Fold 5 IBS: 0.2261815525696507
[I 2024-04-15 16:23:53,942] Trial 45 finished with value: 0.2192325638122877 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 16, 'min_samples_leaf': 14, 'max_depth': 11, 'n_estimators': 369, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.7548177510477919, 'min_weight_fraction_leaf': 0.13669728117144694}. Best is trial 36 with value: 0.2167639286316539.
Fold 1 IBS: 0.2465213089401209
Fold 2 IBS: 0.23046298845066793
Fold 3 IBS: 0.22971391168769142
Fold 4 IBS: 0.24253981917172043
Fold 5 IBS: 0.23157320473775703
[I 2024-04-15 16:24:02,634] Trial 46 finished with value: 0.23616224659759152 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 14, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 314, 'oob_score': True, 'warm_start': True, 'max_features': 1, 'max_samples': 0.8165

Fold 1 IBS: 0.24675120399960646
Fold 2 IBS: 0.2238312246331017
Fold 3 IBS: 0.22373605345231745
Fold 4 IBS: 0.24763583159134048
Fold 5 IBS: 0.2361001387256771
[I 2024-04-15 16:26:18,683] Trial 60 finished with value: 0.23561089048040867 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 20, 'min_samples_leaf': 5, 'max_depth': 5, 'n_estimators': 259, 'oob_score': True, 'warm_start': False, 'max_features': 'sqrt', 'max_samples': 0.5876523474241502, 'min_weight_fraction_leaf': 0.18336534674120722}. Best is trial 53 with value: 0.21672142772101383.
Fold 1 IBS: 0.24544383395694797
Fold 2 IBS: 0.19103401818321838
Fold 3 IBS: 0.1977979215082493
Fold 4 IBS: 0.2360015957855052
Fold 5 IBS: 0.22738640071735247
[I 2024-04-15 16:26:38,480] Trial 61 finished with value: 0.21953275403025466 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 16, 'min_samples_leaf': 3, 'max_depth': 6, 'n_estimators': 363, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.98

Fold 1 IBS: 0.23878866729576506
Fold 2 IBS: 0.19862339947767513
Fold 3 IBS: 0.20582989519828723
Fold 4 IBS: 0.2244448023517984
Fold 5 IBS: 0.2249714920857591
[I 2024-04-15 16:29:31,292] Trial 75 finished with value: 0.218531651281857 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 13, 'min_samples_leaf': 4, 'max_depth': 14, 'n_estimators': 284, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.8355109212222066, 'min_weight_fraction_leaf': 0.1496381039654338}. Best is trial 53 with value: 0.21672142772101383.
Fold 1 IBS: 0.24638426866097177
Fold 2 IBS: 0.22911008901873028
Fold 3 IBS: 0.22825388044344033
Fold 4 IBS: 0.2429147576831134
Fold 5 IBS: 0.2310867513895085
[I 2024-04-15 16:29:40,630] Trial 76 finished with value: 0.23554994943915286 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 17, 'min_samples_leaf': 13, 'max_depth': 10, 'n_estimators': 402, 'oob_score': False, 'warm_start': False, 'max_features': 'log2', 'max_samples': 0.9

Fold 1 IBS: 0.23711736196146355
Fold 2 IBS: 0.20354697260954158
Fold 3 IBS: 0.20782994592008056
Fold 4 IBS: 0.235467444322929
Fold 5 IBS: 0.23168859841356962
[I 2024-04-15 16:31:59,043] Trial 90 finished with value: 0.22313006464551685 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 17, 'min_samples_leaf': 13, 'max_depth': 14, 'n_estimators': 239, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.6267661705142946, 'min_weight_fraction_leaf': 0.049594043056366624}. Best is trial 53 with value: 0.21672142772101383.
Fold 1 IBS: 0.24102633692552217
Fold 2 IBS: 0.19520843787050352
Fold 3 IBS: 0.20356979583511886
Fold 4 IBS: 0.2204498673203032
Fold 5 IBS: 0.22696028171118432
[I 2024-04-15 16:32:10,721] Trial 91 finished with value: 0.21744294393252642 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 20, 'min_samples_leaf': 10, 'max_depth': 13, 'n_estimators': 304, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.

In [60]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [61]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.859
train_ibs:  0.217


#### Test

In [62]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [63]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=13, max_features=None, max_leaf_nodes=19,
                   max_samples=0.9533554910566981,
                   min_weight_fraction_leaf=0.01276054813110837,
                   n_estimators=487, oob_score=True, random_state=123,
                   warm_start=True)

C-index score: 0.506


ExtraSurvivalTrees(max_depth=6, max_features=None, max_leaf_nodes=18,
                   max_samples=0.9463423717095281, min_samples_leaf=4,
                   min_samples_split=14,
                   min_weight_fraction_leaf=0.17857938204264587,
                   n_estimators=273, oob_score=True, random_state=123)

IBS: 0.245


In [64]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis

#### Train

In [65]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-15 16:33:34,409] A new study created in memory with name: no-name-ba565b4f-4594-476b-a7c6-dc8233edf8c9


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-15 16:34:24,189] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-15 16:34:48,259] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-15 16:48:05,999] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.8386796524426539, 'learning_rate': 0.046734492485875676, 'dropout_rate': 0.4821375662037144, 'n_estimators': 402, 'criterion': 'squared_error', 'ccp_alpha': 1.5696007313501796, 'min_weight_fraction_leaf': 0.18684147934268416, 'max_features': 'log2', 'min_impurity_decrease': 1.5044881127471587e-06, 'validation_fraction': 0.8166356053932342, 'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 16, 'max_depth': 4}. Best is trial 9 with value: 0.6859109422281264.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-15 16:49:30,155] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.33235389014851724, 'learning_rate': 0.04522573411670834, 'dropout_rate': 0.2712811374536856, 'n_estimators': 405, 'criterion': 'square

Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-15 17:05:52,605] Trial 25 finished with value: 0.5 and parameters: {'subsample': 0.8502968404151126, 'learning_rate': 0.024443951259730985, 'dropout_rate': 0.2675273969344081, 'n_estimators': 441, 'criterion': 'squared_error', 'ccp_alpha': 2.0753749717266823, 'min_weight_fraction_leaf': 0.3503497788125578, 'max_features': 'auto', 'min_impurity_decrease': 7.237153572123947e-07, 'validation_fraction': 0.41222573804914475, 'min_samples_split': 16, 'max_leaf_nodes': 13, 'min_samples_leaf': 12, 'max_depth': 3}. Best is trial 9 with value: 0.6859109422281264.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-15 17:06:28,988] Trial 26 finished with value: 0.5 and parameters: {'subsample': 0.7670085127536703, 'learning_rate': 0.011828778593594373, 'dropout_rate': 0.4300954216773497, 'n_estimators': 330, 'criterion': 'friedman_mse', 'ccp_alpha':

...

Fold 5 IBS: 0.2293955930480925
[I 2024-04-15 18:31:48,134] Trial 44 finished with value: 0.2359278435123307 and parameters: {'subsample': 0.9992871622667048, 'learning_rate': 0.012272887594565313, 'dropout_rate': 0.11965546339368549, 'n_estimators': 451, 'criterion': 'squared_error', 'ccp_alpha': 0.5504303626596221, 'min_weight_fraction_leaf': 0.10316812300893248, 'max_features': 'auto', 'min_impurity_decrease': 2.2780695231073978e-06, 'validation_fraction': 0.8946516967835402, 'min_samples_split': 17, 'max_leaf_nodes': 14, 'min_samples_leaf': 10, 'max_depth': 2}. Best is trial 41 with value: 0.23454270328649618.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927118
Fold 5 IBS: 0.2293955930480925
[I 2024-04-15 18:32:07,979] Trial 45 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.8888336111822305, 'learning_rate': 0.022080051602678542, 'dropout_rate': 0.2121382569959027, 'n_estimators': 2

Fold 5 IBS: 0.22939559304809248
[I 2024-04-15 18:38:05,268] Trial 55 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.809323041898869, 'learning_rate': 0.0040826970179266685, 'dropout_rate': 0.36065535241543395, 'n_estimators': 435, 'criterion': 'squared_error', 'ccp_alpha': 1.5126136571072866, 'min_weight_fraction_leaf': 0.2002957617254777, 'max_features': 'auto', 'min_impurity_decrease': 2.0868015436948727e-05, 'validation_fraction': 0.8814649292460881, 'min_samples_split': 17, 'max_leaf_nodes': 18, 'min_samples_leaf': 11, 'max_depth': 2}. Best is trial 41 with value: 0.23454270328649618.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.22939559304809248
[I 2024-04-15 18:38:36,279] Trial 56 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.2818850250060197, 'learning_rate': 0.09850922090204048, 'dropout_rate': 0.23297170241739207, 'n_estimators':

Fold 5 IBS: 0.2293955930480925
[I 2024-04-15 18:45:00,080] Trial 66 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.8489175059797514, 'learning_rate': 0.008448226967076113, 'dropout_rate': 0.12725750793823018, 'n_estimators': 424, 'criterion': 'squared_error', 'ccp_alpha': 0.893877768639663, 'min_weight_fraction_leaf': 0.2702972473075093, 'max_features': 'auto', 'min_impurity_decrease': 7.586810829424823e-05, 'validation_fraction': 0.8971088342813041, 'min_samples_split': 17, 'max_leaf_nodes': 17, 'min_samples_leaf': 8, 'max_depth': 4}. Best is trial 62 with value: 0.2343598469061142.
Fold 1 IBS: 0.24724710044658996
Fold 2 IBS: 0.23203988453792296
Fold 3 IBS: 0.22898186806977708
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-15 18:45:36,166] Trial 67 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9594402941587806, 'learning_rate': 0.0039206688180527, 'dropout_rate': 0.25338876644885766, 'n_estimators': 444, '

Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-15 18:51:21,236] Trial 78 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.6708593402641205, 'learning_rate': 0.02742354559286264, 'dropout_rate': 0.12642193192647372, 'n_estimators': 329, 'criterion': 'squared_error', 'ccp_alpha': 0.2456295975030462, 'min_weight_fraction_leaf': 0.3629864105918121, 'max_features': None, 'min_impurity_decrease': 0.00015140034318578482, 'validation_fraction': 0.6173345703129903, 'min_samples_split': 10, 'max_leaf_nodes': 8, 'min_samples_leaf': 11, 'max_depth': 5}. Best is trial 70 with value: 0.2341005897890583.
Fold 1 IBS: 0.2422361212718552
Fold 2 IBS: 0.22571122387519038
Fold 3 IBS: 0.22923653706299701
Fold 4 IBS: 0.2398268927530749
Fold 5 IBS: 0.2262157333144423
[I 2024-04-15 18:51:54,864] Trial 79 finished with value: 0.23264530165551198 and parameters: {'sub

Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977708
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-15 18:55:01,193] Trial 89 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.6555693108457097, 'learning_rate': 0.08348043724232247, 'dropout_rate': 0.1018982518393906, 'n_estimators': 126, 'criterion': 'squared_error', 'ccp_alpha': 5.474477626804302, 'min_weight_fraction_leaf': 0.3922659564335438, 'max_features': None, 'min_impurity_decrease': 1.84353439530822e-05, 'validation_fraction': 0.4550485507485021, 'min_samples_split': 17, 'max_leaf_nodes': 3, 'min_samples_leaf': 6, 'max_depth': 9}. Best is trial 81 with value: 0.23126689637824502.
Fold 1 IBS: 0.24724710044658996
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-15 18:55:08,517] Trial 90 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.6142534760575815, 'lear

In [66]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [67]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.686
train_ibs:  0.231


#### Test

In [68]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [69]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.07426378544613033,
                                 criterion='squared_error',
                                 dropout_rate=0.28125955124323376,
                                 learning_rate=0.013102111413618,
                                 max_features='auto', max_leaf_nodes=16,
                                 min_impurity_decrease=1.4994028685178666e-07,
                                 min_samples_leaf=10,
                                 min_weight_fraction_leaf=0.27579636299120275,
                                 n_estimators=406, random_state=123,
                                 subsample=0.6059965408578151,
                                 validation_fraction=0.6359003593513561)

C-index score: 0.518


GradientBoostingSurvivalAnalysis(ccp_alpha=0.03787089343697578,
                                 criterion='squared_error',
                                 dropout_rate=0.18479264516541616,
                                 learning_rate=0.09426091382038485, max_depth=2,
                                 max_leaf_nodes=6,
                                 min_impurity_decrease=7.4293259533530295e-06,
                                 min_samples_leaf=7, min_samples_split=18,
                                 min_weight_fraction_leaf=0.4036747584607865,
                                 n_estimators=390, random_state=123,
                                 subsample=0.579213280645811,
                                 validation_fraction=0.5325834925366492)

IBS: 0.23


In [70]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [71]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

In [72]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-15 18:57:37,696] A new study created in memory with name: no-name-1e259526-e221-47c2-aac1-135ce1d52b5d


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6374501992031872
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6539923954372624
Fold 5 C-index: 0.6609442060085837
[I 2024-04-15 18:57:43,325] Trial 0 finished with value: 0.6423526693826518 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.6423526693826518.
Fold 1 C-index: 0.6653386454183267
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6425855513307985
Fold 5 C-index: 0.6609442060085837
[I 2024-04-15 18:58:00,042] Trial 1 finished with value: 0.6456489898043869 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 1 with value: 0.6456489898043869.
Fold 1 C-index: 0.6175298804780877
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
Fold 

Fold 1 C-index: 0.6095617529880478
Fold 2 C-index: 0.7015503875968992
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.688212927756654
Fold 5 C-index: 0.6609442060085837
[I 2024-04-15 19:00:37,932] Trial 19 finished with value: 0.6443942804019518 and parameters: {'subsample': 0.38793667853472646, 'dropout_rate': 0.7077875554849441, 'n_estimators': 85, 'learning_rate': 0.04269105711861253}. Best is trial 7 with value: 0.6499035318694023.
Fold 1 C-index: 0.6613545816733067
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6539923954372624
Fold 5 C-index: 0.6609442060085837
[I 2024-04-15 19:00:47,921] Trial 20 finished with value: 0.6471335458766757 and parameters: {'subsample': 0.6004927729383457, 'dropout_rate': 0.608992508329216, 'n_estimators': 310, 'learning_rate': 0.02073176682253709}. Best is trial 7 with value: 0.6499035318694023.
Fold 1 C-index: 0.6613545816733067
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
Fold 

Fold 1 C-index: 0.6613545816733067
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6387832699619772
Fold 5 C-index: 0.6609442060085837
[I 2024-04-15 19:02:51,001] Trial 38 finished with value: 0.6440917207816187 and parameters: {'subsample': 0.7910265829012886, 'dropout_rate': 0.8201500943507289, 'n_estimators': 1, 'learning_rate': 0.0802388198308747}. Best is trial 7 with value: 0.6499035318694023.
Fold 1 C-index: 0.6613545816733067
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6539923954372624
Fold 5 C-index: 0.6609442060085837
[I 2024-04-15 19:02:58,657] Trial 39 finished with value: 0.6471335458766757 and parameters: {'subsample': 0.6483381968153921, 'dropout_rate': 0.6826761322925258, 'n_estimators': 159, 'learning_rate': 0.04656913718567279}. Best is trial 7 with value: 0.6499035318694023.
Fold 1 C-index: 0.6653386454183267
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
Fold 4

...

Fold 3 IBS: 0.27412770407771303
Fold 4 IBS: 0.2250727043053573
Fold 5 IBS: 0.2280063139706162
[I 2024-04-15 19:09:03,969] Trial 19 finished with value: 0.23082306581647552 and parameters: {'subsample': 0.3892284838807411, 'dropout_rate': 0.5354432466556798, 'n_estimators': 131, 'learning_rate': 0.023294697221931306}. Best is trial 7 with value: 0.22271699753958316.
Fold 1 IBS: 0.22813963029627435
Fold 2 IBS: 0.21261919492535192
Fold 3 IBS: 0.2328032377684259
Fold 4 IBS: 0.23177471930860935
Fold 5 IBS: 0.21241390693232276
[I 2024-04-15 19:09:08,457] Trial 20 finished with value: 0.22355013784619687 and parameters: {'subsample': 0.6210873354088753, 'dropout_rate': 0.7933084651006226, 'n_estimators': 66, 'learning_rate': 0.013007963411749002}. Best is trial 7 with value: 0.22271699753958316.
Fold 1 IBS: 0.23294882482388735
Fold 2 IBS: 0.2172731528360657
Fold 3 IBS: 0.23037554967791973
Fold 4 IBS: 0.23393962524579212
Fold 5 IBS: 0.21554329357937044
[I 2024-04-15 19:09:12,801] Trial 21 fini

Fold 4 IBS: 0.22761235729593224
Fold 5 IBS: 0.21391136761511492
[I 2024-04-15 19:10:30,476] Trial 38 finished with value: 0.22439947178405434 and parameters: {'subsample': 0.9323458933659868, 'dropout_rate': 0.13272164755980653, 'n_estimators': 28, 'learning_rate': 0.0802388198308747}. Best is trial 30 with value: 0.22105020743663645.
Fold 1 IBS: 0.22935008117918856
Fold 2 IBS: 0.21153618549256686
Fold 3 IBS: 0.29072451950301204
Fold 4 IBS: 0.2586656669929489
Fold 5 IBS: 0.24321840787686647
[I 2024-04-15 19:10:34,716] Trial 39 finished with value: 0.24669897220891657 and parameters: {'subsample': 0.7076071631229807, 'dropout_rate': 0.36113352605756727, 'n_estimators': 75, 'learning_rate': 0.05598015324329953}. Best is trial 30 with value: 0.22105020743663645.
Fold 1 IBS: 0.4295827306814518
Fold 2 IBS: 0.33729766730917193
Fold 3 IBS: 0.38924638966253877
Fold 4 IBS: 0.31325291352447
Fold 5 IBS: 0.32175848063875884
[I 2024-04-15 19:10:47,759] Trial 40 finished with value: 0.35822763636327

Fold 5 IBS: 0.28019979914084864
[I 2024-04-15 19:12:07,498] Trial 57 finished with value: 0.28649058779241166 and parameters: {'subsample': 0.6825327446543555, 'dropout_rate': 0.5208726510676236, 'n_estimators': 282, 'learning_rate': 0.025565265724993874}. Best is trial 55 with value: 0.22089716088725195.
Fold 1 IBS: 0.2214507245251379
Fold 2 IBS: 0.20506568938296435
Fold 3 IBS: 0.24075353489551313
Fold 4 IBS: 0.2288359521484131
Fold 5 IBS: 0.2104012829571286
[I 2024-04-15 19:12:10,872] Trial 58 finished with value: 0.22130143678183142 and parameters: {'subsample': 0.5793765216559574, 'dropout_rate': 0.869359581862351, 'n_estimators': 38, 'learning_rate': 0.03754893539001649}. Best is trial 55 with value: 0.22089716088725195.
Fold 1 IBS: 0.22332388143407295
Fold 2 IBS: 0.20363163283585628
Fold 3 IBS: 0.24449748198927226
Fold 4 IBS: 0.22907918428885007
Fold 5 IBS: 0.21077017722074096
[I 2024-04-15 19:12:14,443] Trial 59 finished with value: 0.22226047155375853 and parameters: {'subsampl

Fold 5 IBS: 0.2156801492109445
[I 2024-04-15 19:13:24,845] Trial 76 finished with value: 0.2258634284767318 and parameters: {'subsample': 0.6388046490330901, 'dropout_rate': 0.618911653541915, 'n_estimators': 42, 'learning_rate': 0.014880869357715765}. Best is trial 55 with value: 0.22089716088725195.
Fold 1 IBS: 0.22462298117263793
Fold 2 IBS: 0.2073982536472491
Fold 3 IBS: 0.28361457332876805
Fold 4 IBS: 0.24675247717794
Fold 5 IBS: 0.2381675349082399
[I 2024-04-15 19:13:30,812] Trial 77 finished with value: 0.240111164046967 and parameters: {'subsample': 0.541487090871485, 'dropout_rate': 0.7704979082473631, 'n_estimators': 165, 'learning_rate': 0.02189762732192369}. Best is trial 55 with value: 0.22089716088725195.
Fold 1 IBS: 0.22365763185049595
Fold 2 IBS: 0.20591143440013582
Fold 3 IBS: 0.280976248290565
Fold 4 IBS: 0.24848562445786387
Fold 5 IBS: 0.23615081501710733
[I 2024-04-15 19:13:35,987] Trial 78 finished with value: 0.2390363508032336 and parameters: {'subsample': 0.6733

Fold 1 IBS: 0.2179012967955125
Fold 2 IBS: 0.20078159037918683
Fold 3 IBS: 0.26110289322293423
Fold 4 IBS: 0.23412808505560326
Fold 5 IBS: 0.21973897733190245
[I 2024-04-15 19:14:57,131] Trial 96 finished with value: 0.22673056855702786 and parameters: {'subsample': 0.7054650333977128, 'dropout_rate': 0.6262161752789943, 'n_estimators': 61, 'learning_rate': 0.03998698220337849}. Best is trial 81 with value: 0.2208903160925389.
Fold 1 IBS: 0.21963880593620028
Fold 2 IBS: 0.20092268436766023
Fold 3 IBS: 0.2648475542088438
Fold 4 IBS: 0.2331055121729352
Fold 5 IBS: 0.22220093036998428
[I 2024-04-15 19:15:02,102] Trial 97 finished with value: 0.22814309741112476 and parameters: {'subsample': 0.531829760385333, 'dropout_rate': 0.6847533776104763, 'n_estimators': 79, 'learning_rate': 0.03223905987322316}. Best is trial 81 with value: 0.2208903160925389.
Fold 1 IBS: 0.2466414230044532
Fold 2 IBS: 0.23146700010719856
Fold 3 IBS: 0.2289119634694628
Fold 4 IBS: 0.24164291053866815
Fold 5 IBS: 0.

In [73]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [74]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.67
train_ibs:  0.221


#### Test

In [75]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [76]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.9810542276440395,
                                              learning_rate=0.02469841671276124,
                                              n_estimators=1, random_state=123,
                                              subsample=0.2552050789105039)

C-index score: 0.521


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.8767395889398294,
                                              learning_rate=0.030725485144314495,
                                              n_estimators=48, random_state=123,
                                              subsample=0.5967091670188371)

IBS: 0.239


In [77]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

## Results

In [78]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
ExtraSurvivalTrees,0.859,1.0
Randomsurvivalforest,0.826,2.0
GradientBoosting,0.686,3.0
ComponentwiseGradientBoosting,0.670,4.0
CoxElastic,0.641,5.0
CoxRidge,0.635,6.0
CoxLasso,0.562,7.0


In [79]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
Randomsurvivalforest,0.213,1.0
ExtraSurvivalTrees,0.217,2.0
ComponentwiseGradientBoosting,0.221,3.0
GradientBoosting,0.231,4.0
CoxElastic,0.234,5.0
CoxRidge,0.236,6.0
CoxLasso,0.345,7.0


In [80]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
CoxRidge,0.530,1.0
CoxElastic,0.527,2.0
ComponentwiseGradientBoosting,0.521,3.0
GradientBoosting,0.518,4.0
Randomsurvivalforest,0.516,5.0
ExtraSurvivalTrees,0.506,6.0
CoxLasso,0.477,7.0


In [81]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs

,IBS,rank
CoxRidge,0.229,1.5
CoxElastic,0.229,1.5
GradientBoosting,0.230,3.0
ComponentwiseGradientBoosting,0.239,4.0
ExtraSurvivalTrees,0.245,5.0
Randomsurvivalforest,0.258,6.0
CoxLasso,0.464,7.0


In [82]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = 'path_to_your_folder/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']


dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d2/dfs/yeojohnson/no_selection/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

# Modify the file names to match the desired format
modified_file_names = ['d2_dfs_yeojohnson_no_selection_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [83]:
from datetime import date
today = date.today()
print("Date: ", today)

Date:  2024-04-15
